blok: 0

In [1]:
!pip install numpy opencv-python scikit-learn matplotlib rasterio torchsummary tqdm json widgets ipywidgets
#gerekli cuda 5060Tİ için ama 4000 serisi için de çalışır tahminimce
!pip install geopandas fiona
!pip install albumentations
!pip install --pre torch torchvision --extra-index-url https://download.pytorch.org/whl/nightly/cu128


ERROR: Could not find a version that satisfies the requirement json (from versions: none)
ERROR: No matching distribution found for json


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/nightly/cu128


blok: 1 kütüphane

In [2]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import rasterio
from rasterio.features import rasterize
import geopandas as gpd
from shapely.geometry import shape
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import ipywidgets as widgets
from IPython.display import display, clear_output

blok: 2 ekran kartı görünüyormu

In [3]:
print(f"Torch Versiyonu: {torch.__version__}")
if torch.cuda.is_available():
    print(f"Ekran Kartı: {torch.cuda.get_device_name(0)}")
    # Ufak bir test: Rastgele bir sayı üretip GPU'ya atar
    print(f"Çalışma Durumu: {torch.rand(1).cuda().device} (Aktif)")
else:
    print("GPU Bulunamadı.")

Torch Versiyonu: 2.10.0.dev20251124+cu128
Ekran Kartı: NVIDIA GeForce RTX 5060 Ti
Çalışma Durumu: cuda:0 (Aktif)


block: 3 ayarlar ve ne kullanılacak

In [ ]:
CONFIG = {
    "DATA_DIR": "dataset sad/",
    "IMAGE_SUFFIX": "_rgb_nir.tif",   
    "MASK_SUFFIX": "_mask.tif",       
    "MODEL_SAVE_NAME": "gol_bulucu_unet_sentinel.pth",
    "IMG_HEIGHT": 512,         
    "IMG_WIDTH": 512,
    "BATCH_SIZE": 8,
    "LEARNING_RATE": 1e-4,     
    "NUM_EPOCHS": 100,         
    "PATIENCE": 20,            
    
    "IN_CHANNELS": 4,          
    "OUT_CHANNELS": 1,         
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "NUM_WORKERS": 0,
    "PIN_MEMORY": True 
}

print(f"Sistem: {CONFIG['DEVICE']} üzerinde çalışacak.")
torch.backends.cudnn.benchmark = True
print("NVIDIA CUDNN Optimizasyonu: AKTİF")

Sistem: cuda üzerinde çalışacak.
🚀 NVIDIA CUDNN Optimizasyonu: AKTİF


block : 4 maskeyi oluşturacak 

In [5]:

def generate_masks_from_geojson(data_dir):
    # Bu fonksiyon öncekiyle aynı, sadece çıktı kalabalığı azaltıldı
    print("\n--- 1. ADIM: GeoJSON Kontrolü ---")
    files = os.listdir(data_dir)
    tif_files = [f for f in files if f.endswith(CONFIG['IMAGE_SUFFIX'])]
    converted_count = 0
    
    # leave=False: İşlem bitince barı siler
    for tif_file in tqdm(tif_files, desc="Maske Kontrolü", leave=False):
        base_name = tif_file.replace(CONFIG['IMAGE_SUFFIX'], "")
        img_path = os.path.join(data_dir, tif_file)
        mask_name = base_name + CONFIG['MASK_SUFFIX']
        mask_path = os.path.join(data_dir, mask_name)
        
        if os.path.exists(mask_path): continue
            
        possible_jsons = [base_name + ".geojson", base_name + "_json.geojson", base_name + ".json"]
        found_json = next((os.path.join(data_dir, j) for j in possible_jsons if os.path.exists(os.path.join(data_dir, j))), None)
        
        if found_json:
            try:
                with rasterio.open(img_path) as src:
                    meta = src.meta.copy()
                    height, width = src.shape
                    transform = src.transform
                gdf = gpd.read_file(found_json)
                if gdf.empty or gdf.geometry.isnull().all():
                    mask_arr = np.zeros((height, width), dtype=np.uint8)
                else:
                    shapes = ((geom, 255) for geom in gdf.geometry)
                    mask_arr = rasterize(shapes=shapes, out_shape=(height, width), transform=transform, default_value=0, dtype='uint8')
                meta.update(count=1, dtype=rasterio.uint8)
                with rasterio.open(mask_path, 'w', **meta) as dst: dst.write(mask_arr, 1)
                converted_count += 1
            except Exception: pass
                
    if converted_count > 0: print(f"✅ {converted_count} yeni maske oluşturuldu.")
    else: print("ℹ️ Maskeler hazır.")


block : 5 veri setinin işlenmesi 
block : 6 model alanı

In [6]:
import random
from rasterio.windows import Window
import psutil  # RAM kontrolü için

class SatelliteDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None, samples_per_epoch=2000):
        self.transform = transform
        self.samples_per_epoch = samples_per_epoch
        
        self.loaded_images = []
        self.loaded_masks = []
        
        print(f"🚀 Veriler RAM'e yükleniyor... (Bu işlem bir kez yapılır)")
        
        # Tüm büyük resimleri baştan RAM'e alıyoruz
        for img_path, mask_path in zip(image_paths, mask_paths):
            with rasterio.open(img_path) as src:
                # Tüm resmi float32 olarak oku ve normalize et
                img = src.read([1, 2, 3, 4]).transpose(1, 2, 0).astype(np.float32)
                if img.max() > 255.0:
                    img /= 10000.0
                else:
                    img /= 255.0
            
            with rasterio.open(mask_path) as src:
                msk = src.read(1).astype(np.float32)
                msk[msk > 0] = 1.0
            
            self.loaded_images.append(img)
            self.loaded_masks.append(msk)
            
        print(f"✅ {len(self.loaded_images)} adet büyük uydu görüntüsü RAM'e yüklendi.")
        print(f"Kullanılan Sistem RAM'i: {psutil.virtual_memory().percent}%")

    def __len__(self):
        return self.samples_per_epoch

    def __getitem__(self, index):
        # Rastgele bir resim seç (RAM'den)
        img_idx = index % len(self.loaded_images)
        
        # Diskten okumak yerine RAM'deki listeden çekiyoruz (ÇOK HIZLI)
        full_image = self.loaded_images[img_idx]
        full_mask = self.loaded_masks[img_idx]
        
        h, w, _ = full_image.shape
        
        # --- RANDOM CROP İŞLEMİ (RAM ÜZERİNDE) ---
        # Eğer resim 256'dan büyükse rastgele bir yer seç
        if w > CONFIG["IMG_WIDTH"]:
            x_start = random.randint(0, w - CONFIG["IMG_WIDTH"])
        else:
            x_start = 0
            
        if h > CONFIG["IMG_HEIGHT"]:
            y_start = random.randint(0, h - CONFIG["IMG_HEIGHT"])
        else:
            y_start = 0
            
        # Numpy slicing (Çok hızlıdır)
        image_crop = full_image[y_start:y_start+CONFIG["IMG_HEIGHT"], x_start:x_start+CONFIG["IMG_WIDTH"], :]
        mask_crop = full_mask[y_start:y_start+CONFIG["IMG_HEIGHT"], x_start:x_start+CONFIG["IMG_WIDTH"]]
        
        # Eğer kenardaysa ve küçük kaldıysa padding yap
        if image_crop.shape[0] != CONFIG["IMG_HEIGHT"] or image_crop.shape[1] != CONFIG["IMG_WIDTH"]:
             pad_h = CONFIG["IMG_HEIGHT"] - image_crop.shape[0]
             pad_w = CONFIG["IMG_WIDTH"] - image_crop.shape[1]
             image_crop = np.pad(image_crop, ((0, pad_h), (0, pad_w), (0, 0)), mode='constant', constant_values=0)
             mask_crop = np.pad(mask_crop, ((0, pad_h), (0, pad_w)), mode='constant', constant_values=0)

        # Albumentations
        if self.transform:
            augmentations = self.transform(image=image_crop, mask=mask_crop)
            image_crop = augmentations["image"]
            mask_crop = augmentations["mask"]
            
        return image_crop, mask_crop

block : 7 eyitim alanı

In [7]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)

class UNET(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.ups = nn.ModuleList()
        self.downs = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Down part of UNET
        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature

        # Up part of UNET
        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feature*2, feature))

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip_connection = skip_connections[idx//2]

            if x.shape != skip_connection.shape:
                x = torch.nn.functional.interpolate(x, size=skip_connection.shape[2:])

            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx+1](concat_skip)

        return self.final_conv(x)

In [8]:
# %% [markdown]
# block : 7.5 Eğitim Fonksiyonu (Eksik Parça)

# %%
from tqdm import tqdm

def train_fn(loader, model, optimizer, loss_fn, scaler):
    loop = tqdm(loader, desc="Eğitim", leave=False) # İlerleme çubuğu ekledik
    total_loss = 0.0 
    
    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(CONFIG["DEVICE"])
        targets = targets.float().unsqueeze(1).to(CONFIG["DEVICE"])

        # Mixed Precision (Hız ve VRAM tasarrufu için)
        with torch.cuda.amp.autocast():
            predictions = model(data)
            loss = loss_fn(predictions, targets)

        # Backward
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Loss takibi
        total_loss += loss.item()
        
        # Progress bar güncellemesi
        loop.set_postfix(loss=loss.item())
        
    return total_loss / len(loader)

block : 8 ana ksım

In [9]:
if __name__ == "__main__":
    if not os.path.exists(CONFIG["DATA_DIR"]):
        print(f"HATA: '{CONFIG['DATA_DIR']}' klasörü yok.")
    else:
        # Dosya Listesini Al
        files = os.listdir(CONFIG["DATA_DIR"])
        image_files = [f for f in files if f.endswith(CONFIG["IMAGE_SUFFIX"])]
        
        # Sadece dosya yollarını (Path) listeliyoruz, görüntüleri yüklemiyoruz!
        img_paths = []
        msk_paths = []
        
        valid_files_count = 0
        for img_file in image_files:
            base_name = img_file.replace(CONFIG["IMAGE_SUFFIX"], "")
            expected_mask = base_name + CONFIG["MASK_SUFFIX"]
            
            full_img_path = os.path.join(CONFIG["DATA_DIR"], img_file)
            full_msk_path = os.path.join(CONFIG["DATA_DIR"], expected_mask)
            
            if os.path.exists(full_msk_path):
                img_paths.append(full_img_path)
                msk_paths.append(full_msk_path)
                valid_files_count += 1
        
        print(f"Eğitim için {valid_files_count} adet görüntü-maske çifti bulundu.")
        print("NOT: Görüntüler RAM'e yüklenmiyor, eğitim sırasında dinamik olarak kırpılacak (Random Crop).")

        if valid_files_count > 0:
            # Train/Val ayırımı (Dosya bazlı)
            train_img, val_img, train_msk, val_msk = train_test_split(img_paths, msk_paths, test_size=0.1, random_state=42)
            
            # Transformlar (Resize YOK, çünkü zaten kırpıyoruz)
            train_transform = A.Compose([
                A.Rotate(limit=90, p=0.5), # Daha fazla rotasyon
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                ToTensorV2(),
            ])
            val_transform = A.Compose([ToTensorV2()])
            
            # Dataset ve DataLoader
            # samples_per_epoch=2000 -> Her epoch'ta toplam 2000 farklı kare eğitilecek
            train_ds = SatelliteDataset(train_img, train_msk, transform=train_transform, samples_per_epoch=1000)
            val_ds = SatelliteDataset(val_img, val_msk, transform=val_transform, samples_per_epoch=200) # Validasyon için 200 kare
            
            train_loader = DataLoader(
                train_ds, 
                batch_size=CONFIG["BATCH_SIZE"], 
                shuffle=True, 
                num_workers=0, # Windows için 0
                pin_memory=True
            )
            val_loader = DataLoader(
                val_ds, 
                batch_size=CONFIG["BATCH_SIZE"], 
                shuffle=False, 
                num_workers=0,
                pin_memory=True
            )
            
            # Modeli Başlat
            model = UNET(in_channels=CONFIG["IN_CHANNELS"], out_channels=CONFIG["OUT_CHANNELS"]).to(CONFIG["DEVICE"])
            optimizer = optim.Adam(model.parameters(), lr=CONFIG["LEARNING_RATE"])
            loss_fn = nn.BCEWithLogitsLoss()
            scaler = torch.cuda.amp.GradScaler() 
            
            best_loss = float('inf')
            patience_counter = 0
            
            print(f"\n--- 3. ADIM: Eğitim Başlıyor (Gerçek Çözünürlükte) ---")
            
            for epoch in range(CONFIG["NUM_EPOCHS"]):
                model.train()
                train_loss = train_fn(train_loader, model, optimizer, loss_fn, scaler)
                
                model.eval()
                val_loss = 0
                with torch.no_grad():
                    for data, targets in val_loader:
                        data = data.to(CONFIG["DEVICE"])
                        targets = targets.float().unsqueeze(1).to(CONFIG["DEVICE"])
                        preds = model(data)
                        val_loss += loss_fn(preds, targets).item()
                val_loss /= len(val_loader)
                
                # --- KAYIT MANTIĞI (ERROR 1224 Çözümlü) ---
                status_msg = "⏳"
                if val_loss < best_loss:
                    best_loss = val_loss
                    patience_counter = 0
                    try:
                        temp_name = CONFIG["MODEL_SAVE_NAME"] + ".tmp"
                        torch.save(model.state_dict(), temp_name)
                        if os.path.exists(CONFIG["MODEL_SAVE_NAME"]):
                            os.remove(CONFIG["MODEL_SAVE_NAME"])
                        os.rename(temp_name, CONFIG["MODEL_SAVE_NAME"])
                        status_msg = "✅ Kaydedildi"
                    except Exception as e:
                        status_msg = f"❌ Kayıt Hatası"
                else:
                    patience_counter += 1
                    status_msg = f"⚠️ ({patience_counter}/{CONFIG['PATIENCE']})"

                print(f"{epoch+1}/{CONFIG['NUM_EPOCHS']:<8} | {train_loss:.5f}    | {val_loss:.5f}    | {status_msg}")

                if patience_counter >= CONFIG["PATIENCE"]:
                    print("\n🛑 Eğitim bitti (Early Stopping).")
                    break
        else:
            print("HATA: Hiç eşleşen .tif ve _mask.tif bulunamadı.")

Eğitim için 22 adet görüntü-maske çifti bulundu.
NOT: Görüntüler RAM'e yüklenmiyor, eğitim sırasında dinamik olarak kırpılacak (Random Crop).
🚀 Veriler RAM'e yükleniyor... (Bu işlem bir kez yapılır)
✅ 19 adet büyük uydu görüntüsü RAM'e yüklendi.
Kullanılan Sistem RAM'i: 60.7%
🚀 Veriler RAM'e yükleniyor... (Bu işlem bir kez yapılır)
✅ 3 adet büyük uydu görüntüsü RAM'e yüklendi.
Kullanılan Sistem RAM'i: 66.0%


C:\Users\enes\AppData\Local\Temp\ipykernel_20408\1759408854.py:66: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()



--- 3. ADIM: Eğitim Başlıyor (Gerçek Çözünürlükte) ---


Eğitim:   0%|          | 0/16 [00:00<?, ?it/s]C:\Users\enes\AppData\Local\Temp\ipykernel_20408\717160573.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


1/100      | 0.46828    | 0.63093    | ✅ Kaydedildi


2/100      | 0.34317    | 0.57082    | ✅ Kaydedildi


... (72 outputs left out)

75/100      | 0.06008    | 0.07450    | ⚠️ (17/20)


76/100      | 0.04888    | 0.05492    | ⚠️ (18/20)


77/100      | 0.05187    | 0.10344    | ⚠️ (19/20)


78/100      | 0.04670    | 0.05316    | ⚠️ (20/20)

🛑 Eğitim bitti (Early Stopping).
